In [12]:
import numpy as np
from pyscf import ao2mo
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy

In [13]:
print("--- Phase 1: Loading Classical Integrals ---")

# 1. Load your securely saved CASSCF data
data = np.load("hBN_defect_integrals_CASSCF_ONLY.npz")

h1_gd = data['h1_gd']
h2_gd_packed = data['h2_gd']
ecore_gd = data['ecore_gd'].item()

# Unpack the PySCF compressed 2D array into a full 4D tensor for Qiskit
n_active_orbitals = 6
h2_gd = ao2mo.restore(1, h2_gd_packed, n_active_orbitals)

print(f"Loaded Ground State h1 shape: {h1_gd.shape}")
print(f"Restored Ground State h2 shape: {h2_gd.shape}")
print(f"Core Energy Shift: {ecore_gd:.6f} Ha")

# 2. Inject the integrals into Qiskit's Hamiltonian builder
hamiltonian_gd = ElectronicEnergy.from_raw_integrals(h1_a=h1_gd, h2_aa=h2_gd)
hamiltonian_gd.nuclear_repulsion_energy = ecore_gd

# 3. Generate the Second-Quantized Fermionic Operator
fermionic_op_gd = hamiltonian_gd.second_q_op()

print("\n--- Phase 1 Complete ---")
print("Fermionic Operator successfully built!")
print(f"Total Spin-Orbitals (Future Qubits): {fermionic_op_gd.register_length}")

--- Phase 1: Loading Classical Integrals ---
Loaded Ground State h1 shape: (6, 6)
Restored Ground State h2 shape: (6, 6, 6, 6)
Core Energy Shift: -465.791325 Ha

--- Phase 1 Complete ---
Fermionic Operator successfully built!
Total Spin-Orbitals (Future Qubits): 12


In [14]:
print("--- Phase 2: Qubit Mapping & Tapering ---")

# Define the electron spin breakdown from your PySCF Triplet state (6 alpha, 4 beta)
num_particles = (6, 4)
num_spatial_orbitals = 6

# 1. Initialize the Parity Mapper
# Passing the particle number automatically triggers the 2-qubit reduction
mapper = ParityMapper(num_particles=num_particles)

# 2. Translate the Fermionic operator to a Pauli-based Qubit operator
print("Mapping Fermionic operator to Qubits (this may take a few seconds)...")
qubit_op_gd = mapper.map(fermionic_op_gd)

print("\n--- Phase 2 Complete ---")
print(f"Original Spin-Orbitals: {fermionic_op_gd.register_length}")
print(f"Final Qubit Count:      {qubit_op_gd.num_qubits}")
print(f"Total Pauli Strings:    {len(qubit_op_gd)}")

--- Phase 2: Qubit Mapping & Tapering ---
Mapping Fermionic operator to Qubits (this may take a few seconds)...

--- Phase 2 Complete ---
Original Spin-Orbitals: 12
Final Qubit Count:      10
Total Pauli Strings:    1803


In [15]:
print("--- Phase 3: Building Ansatz and Initial State (UCCSD) ---")

# 1. Re-define the Hartree-Fock state for our tapered Hamiltonian
initial_state = HartreeFock(num_spatial_orbitals, num_particles, mapper)

# 2. FIX: particle- and spin-conserving UCCSD ansatz instead of EfficientSU2.
# UCCSD already includes the HF reference internally, so 'ansatz' IS the
# full circuit -- no separate .compose(initial_state) step needed.
ansatz = UCCSD(
    num_spatial_orbitals,
    num_particles,
    mapper,
    initial_state=initial_state,
)
full_circuit = ansatz

print("\n--- Phase 3 Complete (UCCSD) ---")
print(f"Ansatz Parameters: {ansatz.num_parameters}")
print(f"Total Circuit Depth (decomposed): {full_circuit.decompose().depth()}")

--- Phase 3: Building Ansatz and Initial State (UCCSD) ---

--- Phase 3 Complete (UCCSD) ---
Ansatz Parameters: 14
Total Circuit Depth (decomposed): 15


In [17]:
import time
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector
from qiskit_algorithms.optimizers import L_BFGS_B
from qiskit_algorithms import VQE
from scipy.optimize import minimize

print("--- Phase 4: VQE & VQD Execution ---")

# ==========================================
# 4.1: Setup Primitives and Optimizer
# ==========================================
estimator = StatevectorEstimator()
optimizer = L_BFGS_B(maxiter=1000)

# ==========================================
# 4.2: Ground State Optimization (VQE)
# ==========================================
print("\n[1/3] Optimizing Ground State (Root 0) via VQE...")
vqe = VQE(estimator=estimator, ansatz=full_circuit, optimizer=optimizer)
vqe.initial_point = np.zeros(full_circuit.num_parameters)

start_time = time.time()
vqe_result_gd = vqe.compute_minimum_eigenvalue(qubit_op_gd)
time_gd = time.time() - start_time

E_quantum_gd = vqe_result_gd.eigenvalue.real
print(f"VQE Ground State Energy: {E_quantum_gd:.6f} Ha (Took {time_gd:.1f}s)")

# ==========================================
# 4.3: Build Excited State Qubit Operator
# ==========================================
print("\n[2/3] Mapping Excited State Integrals to Qubits...")
h1_ex = data['h1_ex']
h2_ex = ao2mo.restore(1, data['h2_ex'], 6)
ecore_ex = data['ecore_ex'].item()

hamiltonian_ex = ElectronicEnergy.from_raw_integrals(h1_a=h1_ex, h2_aa=h2_ex)
hamiltonian_ex.nuclear_repulsion_energy = ecore_ex
fermionic_op_ex = hamiltonian_ex.second_q_op()

qubit_op_ex = mapper.map(fermionic_op_ex)

# ==========================================
# 4.4: Excited State Optimization (single manual deflation)
# ==========================================
print("\n[3/3] Finding the target excited state via manual deflation...")

def bound_state(circuit, params):
    return Statevector.from_instruction(circuit.assign_parameters(params))

def energy_of(circuit, params, H):
    return bound_state(circuit, params).expectation_value(H).real

def overlap_sq(circuit, params, ref_state):
    return abs(ref_state.inner(bound_state(circuit, params))) ** 2

# Root 0 of the excited-state-geometry Hamiltonian: plain VQE (this is the
# doubly-degenerate ground manifold at this geometry, -15.58 Ha).
vqe_ex0 = VQE(estimator=estimator, ansatz=full_circuit, optimizer=optimizer)
vqe_ex0.initial_point = np.zeros(full_circuit.num_parameters)

start_time = time.time()
res0 = vqe_ex0.compute_minimum_eigenvalue(qubit_op_ex)
time_root0 = time.time() - start_time

E_root0 = res0.eigenvalue.real
theta0 = res0.optimal_point
print(f"Root 0: {E_root0:.6f} Ha (Took {time_root0:.1f}s)")

state0 = bound_state(full_circuit, theta0)

# Root 1 (the target level): deflate away from Root 0. Start from Root 0's
# converged parameters PLUS a random perturbation -- not from zero -- so
# the optimizer doesn't stay trapped in Root 0's own degenerate manifold.
beta = 8.0

def cost_state1(params):
    return energy_of(full_circuit, params, qubit_op_ex) + beta * overlap_sq(full_circuit, params, state0)

rng = np.random.default_rng(42)
x0_state1 = theta0 + rng.normal(scale=0.3, size=full_circuit.num_parameters)

start_time = time.time()
res1 = minimize(cost_state1, x0_state1, method="L-BFGS-B",
                 options={"maxiter": 1000, "ftol": 1e-12, "gtol": 1e-10})
time_root1 = time.time() - start_time

theta1 = res1.x
E_root1 = energy_of(full_circuit, theta1, qubit_op_ex)  # report the pure energy, not the penalized objective
overlap_check = overlap_sq(full_circuit, theta1, state0)
print(f"Root 1 (target excited state): {E_root1:.6f} Ha (Took {time_root1:.1f}s)")
print(f"  sanity check -- residual |<state0|state1>|^2: {overlap_check:.2e} (should be small)")

E_quantum_ex = E_root1

# ==========================================
# Phase 5: Corrected Quantum ZPL Calculation
# ==========================================
print("\n" + "="*50)
print("--- Phase 5: Corrected Quantum ZPL Calculation ---")

ecore_gd = data['ecore_gd'].item()

Total_E_quantum_gd = E_quantum_gd + ecore_gd
Total_E_quantum_ex = E_quantum_ex + ecore_ex

print(f"Total Quantum Ground State:  {Total_E_quantum_gd:.6f} Ha")
print(f"Total Quantum Excited State: {Total_E_quantum_ex:.6f} Ha")

ZPL_quantum_corrected_eV = (Total_E_quantum_ex - Total_E_quantum_gd) * 27.2114

print(f"\nCORRECTED Quantum VQE/VQD ZPL: {ZPL_quantum_corrected_eV:.4f} eV")
print("="*50)

--- Phase 4: VQE & VQD Execution ---

[1/3] Optimizing Ground State (Root 0) via VQE...
VQE Ground State Energy: -14.587562 Ha (Took 1979.0s)

[2/3] Mapping Excited State Integrals to Qubits...

[3/3] Finding the target excited state via manual deflation...
Root 0: -15.580038 Ha (Took 3125.6s)
Root 1 (target excited state): -15.528306 Ha (Took 24502.4s)
  sanity check -- residual |<state0|state1>|^2: 2.32e-09 (should be small)

--- Phase 5: Corrected Quantum ZPL Calculation ---
Total Quantum Ground State:  -480.378887 Ha
Total Quantum Excited State: -480.323869 Ha

CORRECTED Quantum VQE/VQD ZPL: 1.4971 eV


In [18]:
from qiskit_algorithms import NumPyEigensolver

print("--- Exact Classical Diagonalization (Full CI Limit) ---")

# ==========================================
# 1. Exact Ground State
# ==========================================
print("\n[1/2] Diagonalizing Ground State Geometry...")
exact_solver_gd = NumPyEigensolver(k=1)
res_gd = exact_solver_gd.compute_eigenvalues(qubit_op_gd)
exact_E_gd_active = res_gd.eigenvalues[0].real
print(f"Exact Active GS: {exact_E_gd_active:.6f} Ha")

# ==========================================
# 2. Exact Excited State
# ==========================================
print("\n[2/2] Diagonalizing Excited State Geometry...")
exact_solver_ex = NumPyEigensolver(k=3)
res_ex = exact_solver_ex.compute_eigenvalues(qubit_op_ex)
exact_E_ex_active = res_ex.eigenvalues[2].real
print(f"Exact Active ES (Root 2): {exact_E_ex_active:.6f} Ha")

# ==========================================
# 3. Apply Core Corrections & Calculate ZPL
# ==========================================
print("\n" + "="*50)
print("--- Calculating Final ZPL ---")

ecore_gd = data['ecore_gd'].item()
ecore_ex = data['ecore_ex'].item()

Total_Exact_GD = exact_E_gd_active + ecore_gd
Total_Exact_EX = exact_E_ex_active + ecore_ex

print(f"Total Exact Ground State:  {Total_Exact_GD:.6f} Ha")
print(f"Total Exact Excited State: {Total_Exact_EX:.6f} Ha")

ZPL_Exact_eV = (Total_Exact_EX - Total_Exact_GD) * 27.2114
print(f"\nEXACT FULL CI ZPL: {ZPL_Exact_eV:.4f} eV")
print("="*50)

--- Exact Classical Diagonalization (Full CI Limit) ---

[1/2] Diagonalizing Ground State Geometry...
Exact Active GS: -14.587566 Ha

[2/2] Diagonalizing Excited State Geometry...
Exact Active ES (Root 2): -15.528303 Ha

--- Calculating Final ZPL ---
Total Exact Ground State:  -480.378891 Ha
Total Exact Excited State: -480.323866 Ha

EXACT FULL CI ZPL: 1.4973 eV
